In [10]:
import arcpy
import os
import tempfile
import zipfile

# Allow overwrite
arcpy.env.overwriteOutput = True

# List of input geodatabases
gdb_paths = [
    r"D:\Phd Research\Data\FEMA\12090402_SpatialData\Spatial_Files\E_MatagordaBay_1209.gdb",
    r"D:\Phd Research\Data\FEMA\12100401_SpatialData\Central_MatagordaBay.gdb",
    r"D:\Phd Research\Data\FEMA\12100402_SpatialData\W_MatagordaBay.gdb",
    r"D:\Phd Research\Data\FEMA\12100403_SpatialData\Spatial_Files\BLE_FRD_ESAB.gdb",
    r"D:\Phd Research\Data\FEMA\12100405_SpatialData\Spatial_Files\BLE_FRD_AransasBay.gdb",
    r"D:\Phd Research\Data\FEMA\12100407_SpatialData\Spatial_Files\BLE_FRD_Aransas.gdb",
    r"D:\Phd Research\Data\FEMA\12110201_SpatialData\Spatial_Files\BLE_FRD_NCC.gdb",
    r"D:\Phd Research\Data\FEMA\12110202_SpatialData\Spatial_Files\RTO_SouthCCBay_BLE.gdb",
    r"D:\Phd Research\Data\FEMA\12110203_SpatialData\Spatial_Files\RTO_N_Laguna_Madre_BLE.gdb",
    r"D:\Phd Research\Data\FEMA\12110205_SpatialData\Spatial\Baffin_Bay_BLE.gdb",
    r"C:\Users\sahad2\Research_data\FEMA_flood\12090302_SpatialData\Lower_Colorado.gdb",
    r"C:\Users\sahad2\Research_data\FEMA_flood\12100101_SpatialData\Spatial_Files\Lavaca.gdb",
    r"C:\Users\sahad2\Research_data\FEMA_flood\12100102_SpatialData\Spatial_Files\Navidad.gdb",
    r"C:\Users\sahad2\Research_data\FEMA_flood\12100204_SpatialData\12100204_LowerGuadalupe.gdb",
    r"C:\Users\sahad2\Research_data\FEMA_flood\12100303_SpatialData\12100303_SpatialData\BLE_FRD_LSAB.gdb",
    r"C:\Users\sahad2\Research_data\FEMA_flood\12110111_SpatialData\Spatial\BLE_LowerNueces.gdb",
    r"C:\Users\sahad2\Research_data\FEMA_flood\12110204_SpatialData\Spatial_Files\San_Fernando_BLE.gdb"
]

# Name of raster layer in all GDBs
raster_name = "BLE_DEP01PCT"

# Input shapefile ZIP path
boundary_zip = r"D:\Phd Research\GIS\Shape\land_Part_area_utm.zip"

# Final output location
final_output = r"D:\Phd Research\Merged_Raster_UTM14N.tif"

# Temporary workspace
temp_workspace = tempfile.mkdtemp()

# Step 1: Unzip boundary shapefile
with zipfile.ZipFile(boundary_zip, 'r') as zip_ref:
    zip_ref.extractall(temp_workspace)

# Find the .shp file
shp_files = [f for f in os.listdir(temp_workspace) if f.endswith(".shp")]
if not shp_files:
    raise Exception("❌ No shapefile found in the zip.")
boundary_shp = os.path.join(temp_workspace, shp_files[0])

# Step 2: Collect available raster paths
rasters_to_merge = []
for gdb in gdb_paths:
    raster_path = os.path.join(gdb, raster_name)
    if arcpy.Exists(raster_path):
        rasters_to_merge.append(raster_path)
    else:
        print(f"⚠️ Raster not found: {raster_path}")

if not rasters_to_merge:
    raise Exception("❌ No rasters found to merge.")

# Step 3: Mosaic rasters
print("🔄 Merging rasters...")
merged_raster = os.path.join(temp_workspace, "merged_raster.tif")
arcpy.MosaicToNewRaster_management(
    rasters_to_merge,         # input_rasters
    temp_workspace,           # output_location
    "merged_raster.tif",      # output raster name
    "",                       # coordinate system (inherit from first)
    "32_BIT_FLOAT",           # pixel_type
    "",                       # cell_size (let ArcPy decide)
    1,                        # number_of_bands
    "MEAN",                   # mosaic_method
    "FIRST"                   # mosaic_colormap_mode
)

# Step 4: Define projection to EPSG:2278 (Texas South Central)
print("📌 Defining projection...")
arcpy.DefineProjection_management(
    merged_raster,
    arcpy.SpatialReference(2278)
)

# Step 5: Project to WGS 84 / UTM zone 14N (EPSG:32614)
print("📐 Projecting raster to UTM Zone 14N...")
projected_raster = os.path.join(temp_workspace, "projected_raster.tif")
arcpy.ProjectRaster_management(
    merged_raster,
    projected_raster,
    arcpy.SpatialReference(32614),  # UTM 14N
    "BILINEAR"
)

# Step 6: Clip to boundary
print("✂️ Clipping with boundary shapefile...")
arcpy.Clip_management(
    projected_raster,
    rectangle="#",
    out_raster=final_output,
    in_template_dataset=boundary_shp,
    nodata_value="-9999",
    clipping_geometry="ClippingGeometry",
    maintain_clipping_extent="NO_MAINTAIN_EXTENT"
)

print(f"✅ Done. Final clipped raster saved at:\n{final_output}")


🔄 Merging rasters...
📌 Defining projection...
📐 Projecting raster to UTM Zone 14N...
✂️ Clipping with boundary shapefile...
✅ Done. Final clipped raster saved at:
D:\Phd Research\Merged_Raster_UTM14N.tif


In [6]:
import rasterio
from rasterio.merge import merge
from rasterio.mask import mask
import geopandas as gpd
import tempfile
import zipfile
import os
import matplotlib.pyplot as plt

# === Input raster file paths ===
raster_paths = [
    r"C:\Users\sahad2\Research_data\SLR_Depth_Raster_NOAA\TX_Central_slr_depth_3_5ft.tif",
    r"C:\Users\sahad2\Research_data\SLR_Depth_Raster_NOAA\TX_South2_slr_depth_3_5ft.tif",
    #r"C:\Users\sahad2\Research_data\SLR_Depth_Raster_NOAA\TX_North2_slr_depth_3_5ft.tif",
    #r"C:\Users\sahad2\Research_data\SLR_Depth_Raster_NOAA\TX_South1_slr_depth_3_5ft.tif",
    r"C:\Users\sahad2\Research_data\SLR_Depth_Raster_NOAA\TX_North1_slr_depth_3_5ft.tif"
]

# === Land boundary shapefile (zipped) ===
zip_shapefile = r"D:\Phd Research\GIS\Shape\land_Part_area_utm.zip"
output_raster = r"D:\Phd Research\merged_slr_depth_3_5ft_clipped.tif"

# === Step 1: Load all rasters ===
src_files_to_mosaic = [rasterio.open(fp) for fp in raster_paths]
mosaic, out_trans = merge(src_files_to_mosaic)
meta = src_files_to_mosaic[0].meta.copy()
meta.update({
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_trans
})

# === Step 2: Extract shapefile ===
tempdir = tempfile.mkdtemp()
with zipfile.ZipFile(zip_shapefile, 'r') as zip_ref:
    zip_ref.extractall(tempdir)

shp_path = [os.path.join(tempdir, f) for f in os.listdir(tempdir) if f.endswith(".shp")][0]
gdf = gpd.read_file(shp_path)
gdf = gdf.to_crs(meta["crs"])  # reproject to match raster

# === Step 3: Clip merged raster ===
with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as temp_tif:
    temp_unclipped = temp_tif.name

with rasterio.open(temp_unclipped, 'w', **meta) as dst:
    dst.write(mosaic)

with rasterio.open(temp_unclipped) as src:
    out_image, out_transform = mask(src, gdf.geometry, crop=True)
    out_meta = src.meta.copy()
    out_meta.update({
        "height": out_image.shape[1],
        "width": out_image.shape[2],
        "transform": out_transform
    })

with rasterio.open(output_raster, "w", **out_meta) as dest:
    dest.write(out_image)

# === Step 4: Plot for verification ===
plt.figure(figsize=(10, 6))
plt.imshow(out_image[0], cmap='Blues', origin='upper')
plt.title("Merged & Clipped SLR Depth (3.5 ft)")
plt.colorbar(label="Depth (ft)")
plt.tight_layout()
plt.show()

print(f"✅ Merged and clipped raster saved at:\n{output_raster}")


MemoryError: Unable to allocate 125. GiB for an array with shape (1, 198558, 168973) and data type float32